# Authentication & Access Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-1 — OAuth External Auth**: Identity provider configuration, external auth enforcement, kubeadmin removal
- **OCP-2 — Granular RBAC**: ClusterRoles, ClusterRoleBindings, cluster-admin holders, wildcard permissions, self-provisioner status

In [1]:
import os
import sys

# Point to the datastore DB before importing schema (engine binds at import time)
os.environ["OCP_AUDIT_DB"] = os.path.join(
    os.path.dirname(os.path.abspath("__file__")), "..", "datastore", "ocp_audit.db"
)
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "datastore"))

import pandas as pd
from sqlalchemy import func, text

from schema.database import SessionLocal, engine
from schema.models import (
    Cluster,
    ClusterRole,
    ClusterRoleBinding,
    ClusterRoleBindingSubject,
    ClusterRoleRule,
    ClusterRoleRuleApiGroup,
    ClusterRoleRuleNonResourceUrl,
    ClusterRoleRuleResource,
    ClusterRoleRuleVerb,
    OAuthExternalAuth,
    SelfProvisionerBinding,
    SelfProvisionerSubject,
)

session = SessionLocal()
pd.set_option("display.max_colwidth", 80)
print(f"Connected to: {engine.url}")

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


Connected to: sqlite:////home/vagrant/git/openshift-csv-exporter/notebook/../datastore/ocp_audit.db


## Cluster Inventory

In [2]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
df_clusters

3 cluster(s) in dataset


,id,cluster_name,cluster_context,cluster_server
0,1,ocp-prod-east,admin/api-ocp-prod-east:6443,https://api.ocp-prod-east.example.com:6443
1,2,ocp-staging,admin/api-ocp-staging:6443,https://api.ocp-staging.example.com:6443
2,3,ocp-dev,admin/api-ocp-dev:6443,https://api.ocp-dev.example.com:6443


---
## OCP-1: Authentication Posture

Per-cluster summary of external authentication enforcement and kubeadmin account status.

In [3]:
df_auth = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OAuthExternalAuth.external_auth_enforced,
        OAuthExternalAuth.kubeadmin_removed,
        OAuthExternalAuth.identity_providers_count,
    )
    .join(Cluster, OAuthExternalAuth.cluster_id == Cluster.id)
    .distinct()
    .statement,
    engine,
)
print("Authentication posture by cluster")
df_auth

Authentication posture by cluster


,cluster_name,external_auth_enforced,kubeadmin_removed,identity_providers_count
0,ocp-prod-east,True,True,2
1,ocp-staging,True,True,1
2,ocp-dev,False,False,1


### OCP-1: Identity Provider Inventory

All configured identity providers across clusters.

In [4]:
df_idp = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OAuthExternalAuth.idp_name,
        OAuthExternalAuth.idp_type,
        OAuthExternalAuth.idp_mapping_method,
        OAuthExternalAuth.idp_issuer,
        OAuthExternalAuth.access_token_max_age_seconds,
    )
    .join(Cluster, OAuthExternalAuth.cluster_id == Cluster.id)
    .statement,
    engine,
)
print(f"{len(df_idp)} identity provider(s) configured")
df_idp

4 identity provider(s) configured


,cluster_name,idp_name,idp_type,idp_mapping_method,idp_issuer,access_token_max_age_seconds
0,ocp-prod-east,corporate-okta,OpenID,claim,https://corp.okta.com/oauth2/default,28800.0
1,ocp-prod-east,emergency-htpasswd,HTPasswd,claim,NaN,NaN
2,ocp-staging,corporate-okta,OpenID,claim,https://corp.okta.com/oauth2/default,28800.0
3,ocp-dev,dev-htpasswd,HTPasswd,claim,NaN,NaN


### OCP-1: Compliance Flags

Clusters where external auth is **not** enforced or kubeadmin has **not** been removed.

In [5]:
df_flags = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        OAuthExternalAuth.external_auth_enforced,
        OAuthExternalAuth.kubeadmin_removed,
        OAuthExternalAuth.idp_name,
        OAuthExternalAuth.idp_type,
    )
    .join(Cluster, OAuthExternalAuth.cluster_id == Cluster.id)
    .filter(
        (OAuthExternalAuth.external_auth_enforced == False)  # noqa: E712
        | (OAuthExternalAuth.kubeadmin_removed == False)  # noqa: E712
    )
    .statement,
    engine,
)
if df_flags.empty:
    print("All clusters compliant — no findings.")
else:
    print(f"{len(df_flags)} non-compliant finding(s)")
df_flags

1 non-compliant finding(s)


,cluster_name,external_auth_enforced,kubeadmin_removed,idp_name,idp_type
0,ocp-dev,False,False,dev-htpasswd,HTPasswd


---
## OCP-2: Cluster-Admin Holders

All subjects bound to the `cluster-admin` ClusterRole — these have full administrative privileges.

In [6]:
df_admins = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterRoleBinding.binding_name,
        ClusterRoleBindingSubject.subject_kind,
        ClusterRoleBindingSubject.subject_name,
        ClusterRoleBindingSubject.subject_namespace,
    )
    .join(Cluster, ClusterRoleBinding.cluster_id == Cluster.id)
    .join(
        ClusterRoleBindingSubject,
        ClusterRoleBindingSubject.clusterrolebinding_id == ClusterRoleBinding.id,
    )
    .filter(ClusterRoleBinding.role_ref_name == "cluster-admin")
    .order_by(Cluster.cluster_name, ClusterRoleBindingSubject.subject_kind)
    .statement,
    engine,
)
print(f"{len(df_admins)} cluster-admin binding(s) across all clusters")
df_admins

6 cluster-admin binding(s) across all clusters


,cluster_name,binding_name,subject_kind,subject_name,subject_namespace
0,ocp-dev,cluster-admin,Group,system:masters,None
1,ocp-dev,cluster-admin,User,kubeadmin,None
2,ocp-prod-east,cluster-admin,Group,system:masters,None
3,ocp-prod-east,platform-admins-binding,Group,platform-admins,None
4,ocp-prod-east,platform-admins-binding,User,jsmith@example.com,None
5,ocp-staging,cluster-admin,Group,system:masters,None


### OCP-2: Wildcard Permissions

ClusterRoles containing rules with wildcard (`*`) verbs or resources — these grant broad access.

In [7]:
# Roles with wildcard verbs
wild_verbs = (
    session.query(
        Cluster.cluster_name,
        ClusterRole.role_name,
        func.group_concat(ClusterRoleRuleVerb.verb, "; ").label("verbs"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .join(ClusterRoleRule, ClusterRoleRule.clusterrole_id == ClusterRole.id)
    .join(ClusterRoleRuleVerb, ClusterRoleRuleVerb.rule_id == ClusterRoleRule.id)
    .filter(ClusterRoleRuleVerb.verb == "*")
    .group_by(Cluster.cluster_name, ClusterRole.role_name)
)

# Roles with wildcard resources
wild_res = (
    session.query(
        Cluster.cluster_name,
        ClusterRole.role_name,
        func.group_concat(ClusterRoleRuleResource.resource, "; ").label("resources"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .join(ClusterRoleRule, ClusterRoleRule.clusterrole_id == ClusterRole.id)
    .join(
        ClusterRoleRuleResource,
        ClusterRoleRuleResource.rule_id == ClusterRoleRule.id,
    )
    .filter(ClusterRoleRuleResource.resource == "*")
    .group_by(Cluster.cluster_name, ClusterRole.role_name)
)

df_wild_verbs = pd.read_sql(wild_verbs.statement, engine)
df_wild_res = pd.read_sql(wild_res.statement, engine)

df_wildcards = pd.merge(
    df_wild_verbs, df_wild_res, on=["cluster_name", "role_name"], how="outer"
).fillna("")

print(f"{len(df_wildcards)} role(s) with wildcard permissions")
df_wildcards

3 role(s) with wildcard permissions


,cluster_name,role_name,verbs,resources
0,ocp-prod-east,admin,*,*
1,ocp-prod-east,cluster-admin,*,*
2,ocp-staging,cluster-admin,*,*


### OCP-2: Non-Resource URL Access

ClusterRoles granting access to non-resource URLs (API discovery, health endpoints).

In [8]:
df_nru = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterRole.role_name,
        func.group_concat(
            ClusterRoleRuleNonResourceUrl.non_resource_url, "; "
        ).label("non_resource_urls"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .join(ClusterRoleRule, ClusterRoleRule.clusterrole_id == ClusterRole.id)
    .join(
        ClusterRoleRuleNonResourceUrl,
        ClusterRoleRuleNonResourceUrl.rule_id == ClusterRoleRule.id,
    )
    .group_by(Cluster.cluster_name, ClusterRole.role_name)
    .statement,
    engine,
)
print(f"{len(df_nru)} role(s) with non-resource URL permissions")
df_nru

1 role(s) with non-resource URL permissions


,cluster_name,role_name,non_resource_urls
0,ocp-prod-east,system:discovery,/api; /api/*; /apis; /apis/*; /healthz; /version


### OCP-2: Self-Provisioner Status

Self-provisioner bindings control whether users can create their own projects. Clusters **missing** from this table have self-provisioning disabled.

In [9]:
df_sp = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        SelfProvisionerBinding.binding_name,
        SelfProvisionerBinding.role_ref_name,
        SelfProvisionerSubject.subject_kind,
        SelfProvisionerSubject.subject_name,
    )
    .join(Cluster, SelfProvisionerBinding.cluster_id == Cluster.id)
    .join(
        SelfProvisionerSubject,
        SelfProvisionerSubject.binding_id == SelfProvisionerBinding.id,
    )
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)

all_clusters = {c.cluster_name for c in session.query(Cluster).all()}
sp_clusters = set(df_sp["cluster_name"].unique()) if not df_sp.empty else set()
disabled = all_clusters - sp_clusters

print(f"{len(df_sp)} self-provisioner binding(s)")
if disabled:
    print(f"Self-provisioning DISABLED on: {', '.join(sorted(disabled))}")
df_sp

2 self-provisioner binding(s)
Self-provisioning DISABLED on: ocp-dev


,cluster_name,binding_name,role_ref_name,subject_kind,subject_name
0,ocp-prod-east,self-provisioners,self-provisioner,Group,system:authenticated:oauth
1,ocp-staging,self-provisioners,self-provisioner,Group,system:authenticated:oauth


---
### OCP-2: RBAC Summary by Cluster

Aggregated counts per cluster for a high-level view of RBAC scope.

In [10]:
role_counts = (
    session.query(
        Cluster.cluster_name,
        func.count(ClusterRole.id).label("roles"),
    )
    .join(ClusterRole, ClusterRole.cluster_id == Cluster.id)
    .group_by(Cluster.cluster_name)
)

binding_counts = (
    session.query(
        Cluster.cluster_name,
        func.count(ClusterRoleBinding.id).label("bindings"),
    )
    .join(ClusterRoleBinding, ClusterRoleBinding.cluster_id == Cluster.id)
    .group_by(Cluster.cluster_name)
)

admin_counts = (
    session.query(
        Cluster.cluster_name,
        func.count(ClusterRoleBindingSubject.id).label("cluster_admin_subjects"),
    )
    .join(ClusterRoleBinding, ClusterRoleBinding.cluster_id == Cluster.id)
    .join(
        ClusterRoleBindingSubject,
        ClusterRoleBindingSubject.clusterrolebinding_id == ClusterRoleBinding.id,
    )
    .filter(ClusterRoleBinding.role_ref_name == "cluster-admin")
    .group_by(Cluster.cluster_name)
)

df_roles = pd.read_sql(role_counts.statement, engine)
df_bindings = pd.read_sql(binding_counts.statement, engine)
df_admin_c = pd.read_sql(admin_counts.statement, engine)

df_summary = df_roles.merge(df_bindings, on="cluster_name", how="outer").merge(
    df_admin_c, on="cluster_name", how="outer"
).fillna(0)

for col in ["roles", "bindings", "cluster_admin_subjects"]:
    df_summary[col] = df_summary[col].astype(int)

print("RBAC summary by cluster")
df_summary

RBAC summary by cluster


,cluster_name,roles,bindings,cluster_admin_subjects
0,ocp-dev,0,2,2
1,ocp-prod-east,8,7,3
2,ocp-staging,3,3,1


In [11]:
session.close()
print("Session closed.")

Session closed.
